In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pyttsx3
import time
from tensorflow.keras.models import load_model

# ==========================================
# LOAD MODEL
# ==========================================
model = load_model("sign_alphabet_model.keras")

# ==========================================
# LOAD CLASS LABELS
# ==========================================
class_indices = np.load("class_indices.npy", allow_pickle=True).item()
idx_to_class = {v: k for k, v in class_indices.items()}

print("Class Mapping:")
print(idx_to_class)

# ==========================================
# TEXT TO SPEECH
# ==========================================
engine = pyttsx3.init()
last_prediction = ""
last_spoken_time = 0

# Buffer for probability smoothing
prediction_buffer = []

# ==========================================
# MEDIAPIPE HANDS
# ==========================================
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ==========================================
# WEBCAM
# ==========================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# ==========================================
# MAIN LOOP
# ==========================================
while True:
    success, frame = cap.read()
    if not success:
        break

    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        xs, ys = [], []
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            for lm in hand_landmarks.landmark:
                xs.append(lm.x)
                ys.append(lm.y)

        # Bounding box
        x_min, x_max = int(min(xs) * w), int(max(xs) * w)
        y_min, y_max = int(min(ys) * h), int(max(ys) * h)

        padding_x = int((x_max - x_min) * 0.15)
        padding_y = int((y_max - y_min) * 0.15)

        x_min = max(0, x_min - padding_x)
        x_max = min(w, x_max + padding_x)
        y_min = max(0, y_min - padding_y)
        y_max = min(h, y_max + padding_y)

        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

        # Crop hand region
        hand_crop = frame[y_min:y_max, x_min:x_max]

        if hand_crop.size > 0:
            # Resize with aspect ratio preserved
            target_size = 224
            crop_h, crop_w = hand_crop.shape[:2]
            canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
            scale = min(target_size / crop_w, target_size / crop_h)
            new_w, new_h = int(crop_w * scale), int(crop_h * scale)
            resized = cv2.resize(hand_crop, (new_w, new_h))
            x_offset, y_offset = (target_size - new_w) // 2, (target_size - new_h) // 2
            canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
            hand_resized = canvas

            # ==========================================
            # NORMALIZE (rescale=1./255)
            # ==========================================
            cnn_input = hand_resized.astype(np.float32) / 255.0
            cnn_input = np.expand_dims(cnn_input, axis=0)

            # Prediction
            prediction = model.predict(cnn_input, verbose=0)
            confidence = np.max(prediction[0])

            # Probability smoothing
            prediction_buffer.append(prediction[0])
            if len(prediction_buffer) > 10:
                prediction_buffer.pop(0)
            avg_probs = np.mean(prediction_buffer, axis=0)
            stable_class = np.argmax(avg_probs)
            alphabet = idx_to_class[stable_class]

            print(f"Prediction: {alphabet}, Confidence: {confidence:.4f}")
            top5 = np.argsort(avg_probs)[-5:][::-1]
            print("\nTop Predictions:")
            for idx in top5:
                print(f"{idx_to_class[idx]} : {avg_probs[idx]:.4f}")

            cv2.putText(frame, f"Sign: {alphabet} ({confidence:.2f})",
                        (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 255, 0), 2)

            current_time = time.time()
            if alphabet != last_prediction and current_time - last_spoken_time > 2:
                engine.say(alphabet)
                engine.runAndWait()
                last_prediction = alphabet
                last_spoken_time = current_time

            cv2.imshow("CNN Input", hand_resized)

    cv2.imshow("Two-Hand Sign Language Recognition", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()
hands.close()


C:\Users\Inst_\anaconda3\envs\signlang\lib\site-packages\keras\src\saving\saving_lib.py:794: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}


C:\Users\Inst_\anaconda3\envs\signlang\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Prediction: q, Confidence: 0.5573

Top Predictions:
q : 0.5573
g : 0.2010
n : 0.1082
b : 0.0374
f : 0.0329
Prediction: q, Confidence: 0.3449

Top Predictions:
q : 0.4511
g : 0.1998
n : 0.1406
r : 0.0716
m : 0.0437
Prediction: q, Confidence: 0.5766

Top Predictions:
q : 0.3309
g : 0.3254
n : 0.1128
h : 0.0584
r : 0.0569
Prediction: q, Confidence: 0.6297

Top Predictions:
q : 0.2489
g : 0.2441
e : 0.1576
h : 0.1008
n : 0.0847
Prediction: h, Confidence: 0.7740

Top Predictions:
h : 0.2354
q : 0.2014
g : 0.1953
e : 0.1423
n : 0.0679
Prediction: h, Confidence: 0.5858

Top Predictions:
h : 0.2938
e : 0.1732
q : 0.1686
g : 0.1627
n : 0.0566
Prediction: h, Confidence: 0.8106

Top Predictions:
h : 0.3677
e : 0.1588
q : 0.1450
g : 0.1395
n : 0.0485
Prediction: h, Confidence: 0.3843

Top Predictions:
h : 0.3606
e : 0.1870
q : 0.1271
g : 0.1221
l : 0.0599
Prediction: h, Confidence: 0.6740

Top Predictions:
h : 0.3954
e : 0.1874
q : 0.1131
g : 0.1085
l : 0.0661
Prediction: h, Confidence: 0.5574

To

In [4]:
import cv2
import mediapipe as mp
import numpy as np
import pyttsx3
import time
import os
from tensorflow.keras.models import load_model

# ==========================================
# LOAD MODEL
# ==========================================
model = load_model("sign_alphabet_model.keras")

# ==========================================
# LOAD CLASS LABELS
# ==========================================
class_indices = np.load("class_indices.npy", allow_pickle=True).item()
idx_to_class = {v: k for k, v in class_indices.items()}

print("Class Mapping:")
print(idx_to_class)

# ==========================================
# TEXT TO SPEECH
# ==========================================
engine = pyttsx3.init()
last_prediction = ""
last_spoken_time = 0

# Buffer for probability smoothing
prediction_buffer = []

# ==========================================
# MEDIAPIPE HANDS
# ==========================================
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# ==========================================
# WEBCAM
# ==========================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# ==========================================
# CREATE FOLDER FOR UNCERTAIN CROPS
# ==========================================
save_dir = "uncertain_crops"
os.makedirs(save_dir, exist_ok=True)

# ==========================================
# MAIN LOOP
# ==========================================
while True:
    success, frame = cap.read()
    if not success:
        break

    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        xs, ys = [], []
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            for lm in hand_landmarks.landmark:
                xs.append(lm.x)
                ys.append(lm.y)

        # Bounding box
        x_min, x_max = int(min(xs) * w), int(max(xs) * w)
        y_min, y_max = int(min(ys) * h), int(max(ys) * h)

        padding_x = int((x_max - x_min) * 0.15)
        padding_y = int((y_max - y_min) * 0.15)

        x_min = max(0, x_min - padding_x)
        x_max = min(w, x_max + padding_x)
        y_min = max(0, y_min - padding_y)
        y_max = min(h, y_max + padding_y)

        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)

        # Crop hand region
        hand_crop = frame[y_min:y_max, x_min:x_max]

        if hand_crop.size > 0:
            # Resize with aspect ratio preserved
            target_size = 224
            crop_h, crop_w = hand_crop.shape[:2]
            canvas = np.zeros((target_size, target_size, 3), dtype=np.uint8)
            scale = min(target_size / crop_w, target_size / crop_h)
            new_w, new_h = int(crop_w * scale), int(crop_h * scale)
            resized = cv2.resize(hand_crop, (new_w, new_h))
            x_offset, y_offset = (target_size - new_w) // 2, (target_size - new_h) // 2
            canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
            hand_resized = canvas

            # Normalize (rescale=1./255)
            cnn_input = hand_resized.astype(np.float32) / 255.0
            cnn_input = np.expand_dims(cnn_input, axis=0)

            # Prediction
            prediction = model.predict(cnn_input, verbose=0)
            confidence = np.max(prediction[0])

            # Probability smoothing
            prediction_buffer.append(prediction[0])
            if len(prediction_buffer) > 10:
                prediction_buffer.pop(0)
            avg_probs = np.mean(prediction_buffer, axis=0)
            stable_class = np.argmax(avg_probs)
            alphabet = idx_to_class[stable_class]

            print(f"Prediction: {alphabet}, Confidence: {confidence:.4f}")
            top5 = np.argsort(avg_probs)[-5:][::-1]
            print("\nTop Predictions:")
            for idx in top5:
                print(f"{idx_to_class[idx]} : {avg_probs[idx]:.4f}")

            cv2.putText(frame, f"Sign: {alphabet} ({confidence:.2f})",
                        (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX,
                        1, (0, 255, 0), 2)

            # ==========================================
            # SAVE UNCERTAIN CROPS
            # ==========================================
            if confidence < 0.7:
                filename = os.path.join(save_dir, f"uncertain_{int(time.time()*1000)}.jpg")
                cv2.imwrite(filename, hand_resized)
                print(f"Saved uncertain crop: {filename}")

            # Text-to-speech
            current_time = time.time()
            if alphabet != last_prediction and current_time - last_spoken_time > 2 and confidence > 0.7:
                engine.say(alphabet)
                engine.runAndWait()
                last_prediction = alphabet
                last_spoken_time = current_time

            cv2.imshow("CNN Input", hand_resized)

    cv2.imshow("Two-Hand Sign Language Recognition", frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()
hands.close()


Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}
Prediction: g, Confidence: 0.5678

Top Predictions:
g : 0.5678
h : 0.1780
j : 0.1034
w : 0.0367
f : 0.0331
Saved uncertain crop: uncertain_crops\uncertain_1779890786280.jpg
Prediction: g, Confidence: 0.7762

Top Predictions:
g : 0.6720
h : 0.1293
j : 0.0612
n : 0.0308
q : 0.0245
Prediction: g, Confidence: 0.8891

Top Predictions:
g : 0.7444
h : 0.0904
j : 0.0417
q : 0.0298
n : 0.0274
Prediction: g, Confidence: 0.7800

Top Predictions:
g : 0.7533
h : 0.0792
j : 0.0389
q : 0.0321
n : 0.0311
Prediction: g, Confidence: 0.3941

Top Predictions:
g : 0.6027
h : 0.1422
e : 0.0783
j : 0.0403
q : 0.0349
Saved uncertain crop: uncertain_crops\uncertain_1779890786974.jpg
Prediction: g, Confidence: 0.5376

Top Predictions:
g : 0.5023
h : 0.2081
e : 0.1146
j : 0.03